Dihedral angle and ramachandran plot analysis

In [ ]:
import gsd.hoomd
import numpy as np
import freud
import matplotlib.pyplot as plt
import pandas as pd

def analyzedata():
    # Load trajectory
    traj_path = '/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/trajectory_bias_1.0.gsd'
    bcd = gsd.hoomd.open(traj_path, 'r')
    traj = bcd[10000:20000]
    frame = traj[0]

    # Build tables for psi and phi dihedrals
    table_psi = {}
    table_phi = {}
    dihedral_types = frame.dihedrals.types
    dihedrals_psi = frame.dihedrals.typeid == dihedral_types.index('psi')
    dihedrals_phi = frame.dihedrals.typeid == dihedral_types.index('phi')

    for idx, group in enumerate(frame.dihedrals.group):
        if dihedrals_psi[idx]:
            table_psi[group[1]] = group
        if dihedrals_phi[idx]:
            table_phi[group[2]] = group

    # Remove mismatched dihedrals
    mismatched_keys = table_phi.keys() ^ table_psi.keys()
    for key in mismatched_keys:
        table_psi.pop(key, None)
        table_phi.pop(key, None)

    # Initialize data storage (phi, psi)
    timeseries = {k: np.zeros((len(traj), 2)) for k in table_psi}

    # Dihedral angle calculator
    def compute_dihedral(array):
        b1 = array[1] - array[0]
        b2 = array[2] - array[1]
        b3 = array[3] - array[2]

        b2 /= np.linalg.norm(b2)
        n1 = np.cross(b1, b2)
        n1 /= np.linalg.norm(n1)
        n2 = np.cross(b2, b3)
        n2 /= np.linalg.norm(n2)
        m1 = np.cross(n1, b2)

        x = np.dot(n1, n2)
        y = np.dot(m1, n2)
        return -np.arctan2(y, x)

    # Compute dihedral angles
    for t, frame in enumerate(traj):
        freud_box = freud.box.Box.from_box(frame.configuration.box)
        unwrapped = freud_box.unwrap(frame.particles.position, frame.particles.image)

        for key in table_psi.keys():
            psi_group = table_psi[key]
            phi_group = table_phi[key]

            psi = compute_dihedral(unwrapped[psi_group])
            phi = compute_dihedral(unwrapped[phi_group])

            timeseries[key][t] = [phi, psi]

    # Collect all angles into lists
    all_phi = []
    all_psi = []
    for key in timeseries:
        angles = timeseries[key]
        all_phi.extend(np.degrees(angles[:, 0]))
        all_psi.extend(np.degrees(angles[:, 1]))

    # Optional: save CSV
    df = pd.DataFrame({"phi_deg": all_phi, "psi_deg": all_psi})
    csv_name = "dihedral_angles_1.0.csv"
    df.to_csv(csv_name, index=False)
    print(f" Saved dihedral angles to {csv_name}")

    # -------------------------
    #  RAMACHANDRAN PLOT
    # -------------------------
    plt.figure(figsize=(12, 10))
    hb = plt.hexbin(all_phi, all_psi, gridsize=100, cmap='viridis', bins='log')
    cbar = plt.colorbar(hb)
    cbar.set_label('Count (log scale)', fontsize=26, fontweight='bold')
    cbar.ax.tick_params(labelsize=14)
    for t in cbar.ax.get_yticklabels():
        t.set_fontweight('bold')
    plt.xlabel('Phi angle (degrees)', fontsize=30, fontweight="bold")
    plt.ylabel('Psi angle (degrees)', fontsize=30, fontweight="bold")
    plt.title('Ramachandran Plot', fontsize=26,fontweight='bold')
    plt.xlim(-180, 180)
    plt.ylim(-180, 180)
    plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(0, color='black', linestyle='--', linewidth=0.5)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xticks(fontsize=26, fontweight='bold')
    plt.yticks(fontsize=26, fontweight='bold')
    plt.tight_layout()
    plt.savefig("ramachandran_plot_1.0.png", dpi=600)
    plt.show()

# Run analysis
analyzedata()


Helical Propensity Calculation From Simulation Trajectory

In [ ]:
import gsd.hoomd
import numpy as np
import freud
import math
import matplotlib.pyplot as plt
from matplotlib import rcParams
import os
import pandas as pd

def analyzedata_sets(sets):
    rcParams['font.family'] = 'serif'
    rcParams['font.size'] = 12

    bias_values = np.linspace(0.75, 1.0, 12)

    for dataset in sets:
        set_name = dataset["name"]
        folder_path = dataset["base_path"]
        output_folder = os.path.join(set_name)
        os.makedirs(output_folder, exist_ok=True)

        print(f"\n📂 Processing dataset: {set_name}")
        avg_helicity_per_bias = []

        for bias in bias_values:
            # Handle filenames
            if bias in [0.75, 1.0]:
                trajectory_name = os.path.join(folder_path, f"trajectory_bias_{bias}.gsd")
            else:
                trajectory_name = os.path.join(folder_path, f"trajectory_bias_{bias:.16f}.gsd")

            if not os.path.isfile(trajectory_name):
                print(f"⚠️  File {trajectory_name} not found. Skipping.")
                continue

            print(f"  Processing bias {bias:.2f} -> {trajectory_name}")
            bcd = gsd.hoomd.open(trajectory_name, 'r')
            traj = bcd[10000:20000]
            frame = traj[0]

            N_particles = frame.particles.N
            N_chains = 64
            particles_per_chain = N_particles // N_chains

            # Build dihedral tables
            tables_psi = [dict() for _ in range(N_chains)]
            tables_phi = [dict() for _ in range(N_chains)]

            dihedral_types = frame.dihedrals.types
            dihedrals_psi = frame.dihedrals.typeid == dihedral_types.index('psi')
            dihedrals_phi = frame.dihedrals.typeid == dihedral_types.index('phi')

            for idx, group in enumerate(frame.dihedrals.group):
                chain_idx = group[0] // particles_per_chain
                if dihedrals_psi[idx]:
                    tables_psi[chain_idx][group[1]] = group
                if dihedrals_phi[idx]:
                    tables_phi[chain_idx][group[2]] = group

            # Remove mismatches
            for chain in range(N_chains):
                mismatched_keys = tables_phi[chain].keys() ^ tables_psi[chain].keys()
                for key in mismatched_keys:
                    tables_psi[chain].pop(key, None)
                    tables_phi[chain].pop(key, None)

            # Store dihedrals
            all_timeseries = [{k: np.zeros((len(traj), 2)) for k in tables_psi[chain].keys()} 
                              for chain in range(N_chains)]
            gyration_time = np.zeros((len(traj), N_chains))

            def compute_dihedral(array):
                b1 = array[1, :] - array[0, :]
                b2 = array[2, :] - array[1, :]
                b3 = array[3, :] - array[2, :]
                b2 /= np.linalg.norm(b2)
                n1 = np.cross(b1, b2); n1 /= np.linalg.norm(n1)
                n2 = np.cross(b2, b3); n2 /= np.linalg.norm(n2)
                m1 = np.cross(n1, b2)
                x = np.dot(n1, n2); y = np.dot(m1, n2)
                return -np.arctan2(y, x)

            for time, frame in enumerate(traj):
                freud_box = freud.box.Box.from_box(frame.configuration.box)
                unwrapped = freud_box.unwrap(frame.particles.position, frame.particles.image)

                for chain in range(N_chains):
                    start = chain * particles_per_chain
                    end = (chain + 1) * particles_per_chain
                    chain_pos = frame.particles.position[start:end]
                    chain_mass = frame.particles.mass[start:end]
                    com = np.sum(chain_mass[:, None] * chain_pos, axis=0) / np.sum(chain_mass)
                    gyration_time[time, chain] = np.sqrt(np.sum(chain_mass * np.sum((chain_pos - com)**2, axis=1)) / np.sum(chain_mass))

                    for key in tables_psi[chain].keys():
                        psi_g = tables_psi[chain][key]
                        phi_g = tables_phi[chain][key]
                        psi = compute_dihedral(unwrapped[psi_g, :])
                        phi = compute_dihedral(unwrapped[phi_g, :])
                        all_timeseries[chain][key][time] = np.array([phi, psi])

            # Helicity detection
            def is_helical_t(angles):
                helix_phi = (angles[:, 0] > math.radians(-160)) & (angles[:, 0] < math.radians(-20))
                helix_psi = (angles[:, 1] > math.radians(-120)) & (angles[:, 1] < math.radians(50))
                return helix_phi & helix_psi

            all_timeseries_helix = []
            for chain in range(N_chains):
                timeseries = all_timeseries[chain]
                all_keys = list(timeseries.keys())
                valid_helix_keys = all_keys[1:-1]
                timeseries_helix = {}
                for res, key in enumerate(valid_helix_keys):
                    left = all_keys[all_keys.index(key) - 1]
                    right = all_keys[all_keys.index(key) + 1]
                    my_angles_h = is_helical_t(timeseries[key])
                    my_left_angles_h = is_helical_t(timeseries[left])
                    my_right_angles_h = is_helical_t(timeseries[right])
                    is_helix = my_angles_h & my_left_angles_h & my_right_angles_h
                    timeseries_helix[res+2] = is_helix
                all_timeseries_helix.append(timeseries_helix)

            # Propensity calculation
            num_residues = len(all_timeseries_helix[0])
            residues = list(range(2, num_residues + 2))
            chain_propensities = np.zeros((N_chains, num_residues))
            for chain in range(N_chains):
                for residue, helicities in all_timeseries_helix[chain].items():
                    res_idx = residue - 2
                    chain_propensities[chain, res_idx] = np.mean(helicities)

            avg_propensities = np.mean(chain_propensities, axis=0)
            std_propensities = np.std(chain_propensities, axis=0)

            # Save per-residue CSV
            df_prop = pd.DataFrame({
                "Residue": residues,
                "Average Propensity": avg_propensities,
                "Standard Deviation": std_propensities
            })
            df_prop.to_csv(os.path.join(output_folder, f'propensities_{bias:.2f}.csv'), index=False)

            # Plot per-residue helicity
            plt.figure(figsize=(12, 6))
            plt.errorbar(residues, avg_propensities, yerr=std_propensities, fmt='o-', markersize=5,
                         capsize=3, color='green', ecolor='lightgray', elinewidth=2)
            plt.xlabel('Residue Number')
            plt.ylabel('Helicity Propensity')
            plt.title(f'{set_name}: Helicity Propensity (Bias {bias:.2f})')
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'propensities_{bias:.2f}.png'), dpi=300)
            plt.close()

            # Chain helicity with per-chain STD
            chain_helicity_mean = np.mean(chain_propensities, axis=1)
            chain_helicity_std = np.std(chain_propensities, axis=1)
            df_chain = pd.DataFrame({
                "Chain": np.arange(N_chains),
                "Average_Helicity": chain_helicity_mean,
                "STD_Helicity": chain_helicity_std
            })
            df_chain.to_csv(os.path.join(output_folder, f"chain_helicity_bias_{bias:.2f}.csv"), index=False)

            # Bar plot
            plt.figure(figsize=(14, 6))
            x_pos = np.arange(N_chains)
            plt.bar(x_pos, chain_helicity_mean, yerr=chain_helicity_std,
                    color='skyblue', error_kw=dict(elinewidth=1, ecolor='red', capsize=3))
            plt.axhline(y=np.mean(chain_helicity_mean), color='red', linestyle='--', label=f'Average: {np.mean(chain_helicity_mean):.3f}')
            plt.xlabel('Chain Index')
            plt.ylabel('Average Helicity Propensity')
            plt.title(f'{set_name}: Helicity Across Chains (Bias {bias:.2f})')
            plt.xticks(x_pos[::5])
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6, axis='y')
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'chain_helicity_{bias:.2f}.png'), dpi=300)
            plt.close()

            avg_helicity_per_bias.append([bias, np.mean(chain_helicity_mean), np.std(chain_helicity_mean)])

        # Final plot: avg helicity vs bias
        df_avg = pd.DataFrame(avg_helicity_per_bias, columns=["Bias", "Average_Helicity", "STD_Helicity"])
        df_avg.to_csv(os.path.join(output_folder, "average_helicity_vs_bias.csv"), index=False)

        plt.figure(figsize=(10, 6))
        plt.errorbar(df_avg["Bias"], df_avg["Average_Helicity"], yerr=df_avg["STD_Helicity"],
                     marker='o', capsize=4, linewidth=2)
        plt.xlabel("Bias")
        plt.ylabel("Average Helicity Propensity")
        plt.title(f"{set_name}: Average Helicity vs Bias")
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, "average_helicity_vs_bias.png"), dpi=300)
        plt.close()
        print(f" Done with dataset '{set_name}'. Results saved in folder '{output_folder}'.")

# === Define sets ===
sets = [
    {"name": "Full_length", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/"},
    {"name": "H0_H3", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/3de12d848c06a70b5668c64369550562/"},
    {"name": "H4_H6", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/7fd5f713b9bd4167bafd3d2d9378bc51/"}
]

# Run the analysis
analyzedata_sets(sets)


Residue level Propensity Plot

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Folder with your CSV files
folder_path = "Code_simulation_analysis/analysis/helical_propensity/Full_length/"

# List of files to load
files = [
    "propensities_1.00.csv",
    "propensities_0.98.csv",
    "propensities_0.95.csv",
    "propensities_0.93.csv",
    "propensities_0.91.csv",
    "propensities_0.89.csv",
    "propensities_0.86.csv",
    "propensities_0.84.csv",
    "propensities_0.82.csv",
    "propensities_0.80.csv",
    "propensities_0.77.csv",
    "propensities_0.75.csv"
    
]

# Extract % labels from filenames
labels = [str(int(float(f.split('_')[1].replace('.csv', '')) * 100)) for f in files]

# Colormap
colors = plt.cm.viridis(np.linspace(0, 1, len(files)))

# Create figure
fig, ax = plt.subplots(figsize=(12, 7))

# Plot all datasets
for file, color, label in zip(files, colors, labels):
    file_path = os.path.join(folder_path, file)

    if not os.path.exists(file_path):
        print(f"⚠ File not found: {file}")
        continue

    data = pd.read_csv(file_path)

    residue = data["Residue"]
    avg_propensity = data["Average Propensity"]
    std_dev = data["Standard Deviation"]

    ax.errorbar(
        residue,
        avg_propensity,
        yerr=std_dev,
        fmt='-',
        color=color,
        label=f'{label}%',
        capsize=3,
        linewidth=2
    )

# ----------------------------
# Highlight Structural Segments (H0–H6)
# ----------------------------
segments = [
    (3, 22, "H0"),
    (26, 80, "H1"),
    (84, 136, "H2"),
    (137, 156, "H3"),
    (164, 188, "H4"),
    (192, 215, "H5"),
    (251, 264, "H6")
]

# Get y-limits after plotting
y_min, y_max = ax.get_ylim()
y_text = y_max * 0.97  # label position near top

for start, end, label in segments:
    # Light shading
    ax.axvspan(start, end, color='grey', alpha=0.12)

    # Centered segment label
    ax.text(
        (start + end) / 2,
        y_text,
        label,
        ha='center',
        va='center',
        fontsize=16,
        fontweight='bold',
        color='black'
    )

# ----------------------------
# Formatting
# ----------------------------
ax.set_xlabel("Residue Index", fontsize=30, fontweight="bold")
ax.set_ylabel("Helical Propensity", fontsize=30, fontweight="bold")
#ax.set_title("IM30", fontsize=22, fontweight="bold")

# Bold and large ticks
ax.tick_params(axis='both', labelsize=26)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

# Bold legend
legend = ax.legend(title="H-bond Strength", fontsize=14)
plt.setp(legend.get_title(), fontsize=16, fontweight='bold')
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()

# Save figure
output_path = "IM30_helical_propensity_plot_segmented_plasma.png"
plt.savefig(output_path, dpi=600)
print(f"Plot saved to: {output_path}")

plt.show()


Average Helicity Plot With HBS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Paths to the folders
folders = [
    "Code_simulation_analysis/analysis/helical_propensity/full_length/",
    "Code_simulation_analysis/analysis/helical_propensity/H0_3/",
    "Code_simulation_analysis/analysis/helical_propensity/H4_6/"
]

labels = ['IM30', 'IM30 H0-3', 'IM30 H4-6']
colors = ['red', 'green', 'blue']

filename = "average_helicity_vs_bias.csv"

fig, ax = plt.subplots(figsize=(9, 7))

for folder, label, color in zip(folders, labels, colors):
    file_path = os.path.join(folder, filename)
    
    data = pd.read_csv(file_path)
    
    bias = data['Bias'] * 100  # Convert to percentage
    avg_helicity = data['Average_Helicity']
    std_helicity = data['STD_Helicity']
    
    ax.errorbar(
        bias,
        avg_helicity,
        yerr=std_helicity,
        fmt='d-',
        color=color,
        label=label,
        capsize=4,
        linewidth=2.5,
        markersize=6
    )

# ----------------------------
# Formatting
# ----------------------------

ax.set_xlabel('H-bond Strength (%)', fontsize=30, fontweight='bold')
ax.set_ylabel('Helical Propensity', fontsize=30, fontweight='bold')

ax.tick_params(axis='both', labelsize=26)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

ax.set_ylim(0, 1.0)

# Proper bold legend
legend = ax.legend(loc='upper left', fontsize=26, frameon=False)
for text in legend.get_texts():
    text.set_fontweight('bold')
# Full box with thicker border
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()

# Save figure
output_path = 'Code_simulation_analysis/analysis/helical_propensity/average_helicity_vs_bias_plot.png'
plt.savefig(output_path, dpi=600)
print(f"Plot saved to: {output_path}")

plt.show()


Helicity Plot at 100 % HBS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Base folder and file
base_path = "Code_simulation_analysis/analysis/helical_propensity/"
filename = "propensities_1.00.txt"

# Folders and labels
folders = {
    "IM30": "Full_length",
    "IM30 H0-3": "H0_H3",
    "IM30 H4-6": "H4_H6"
}

colors = ['r', 'g', 'b']
start_H4_6 = 157  # Adjust numbering for H4-6 fragment

fig, ax = plt.subplots(figsize=(12, 7))

# Plot datasets
for (label, folder), color in zip(folders.items(), colors):
    data = pd.read_csv(os.path.join(base_path, folder, filename), sep='\t')
    residue = data['Residue']
    
    # Adjust residue numbering for H4-6 fragment
    if folder == "H4_6":
        residue = residue - residue.iloc[0] + start_H4_6
    
    ax.errorbar(
        residue,
        data['Average Propensity'],
        yerr=data['Standard Deviation'],
        fmt='-',
        color=color,
        capsize=3,
        linewidth=2.5,
        label=label
    )

# ----------------------------
# Highlight Structural Segments (H0–H6)
# ----------------------------
segments = [
    (3, 22, "H0"),
    (26, 80, "H1"),
    (84, 136, "H2"),
    (137, 156, "H3"),
    (164, 188, "H4"),
    (192, 215, "H5"),
    (251, 264, "H6")
]

y_min, y_max = ax.get_ylim()
y_text = y_max * 0.97

for start, end, label in segments:
    ax.axvspan(start, end, color='grey', alpha=0.10)
    ax.text(
        (start + end) / 2,
        y_text,
        label,
        ha='center',
        va='center',
        fontsize=26,
        fontweight='bold'
    )

# ----------------------------
# Formatting
# ----------------------------
ax.set_xlabel('Residue Index', fontsize=30, fontweight='bold')
ax.set_ylabel('Helical Propensity', fontsize=30, fontweight='bold')

ax.tick_params(axis='both', labelsize=26)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

legend = ax.legend(frameon=False, fontsize=26)
for text in legend.get_texts():
    text.set_fontweight('bold')
# Full box with thicker border
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()

plt.savefig(os.path.join(base_path, 'combined_100_percent_plot_segmented.png'), dpi=600)
plt.show()


Helicity vs HBS for Fragments

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Folder containing full-length propensity files
folder_path = "Code_simulation_analysis/analysis/helical_propensity/Full_length/"

files = [
    "propensities_1.00.csv",
    "propensities_0.98.csv",
    "propensities_0.95.csv",
    "propensities_0.93.csv",
    "propensities_0.91.csv",
    "propensities_0.89.csv",
    "propensities_0.86.csv",
    "propensities_0.84.csv",
    "propensities_0.82.csv",
    "propensities_0.80.csv",
    "propensities_0.77.csv",
    "propensities_0.75.csv"
]

# Extract H-bond strength from filenames
hbs_values = [float(f.split('_')[1].replace('.csv', '')) * 100 for f in files]

# Define segments
segments = {
    "H0": (3, 22),
    "H1": (26, 80),
    "H2": (84, 136),
    "H3": (137, 156),
    "H4": (164, 188),
    "H5": (192, 215),
    "H6": (251, 264)
}

# Store mean and SEM
segment_means = {key: [] for key in segments.keys()}
segment_sems  = {key: [] for key in segments.keys()}

# ----------------------------
# Extract segment statistics
# ----------------------------
for file in files:
    file_path = os.path.join(folder_path, file)
    
    if not os.path.exists(file_path):
        print(f"⚠ File not found: {file}")
        continue
    
    data = pd.read_csv(file_path)
    
    for seg_name, (start, end) in segments.items():
        seg_data = data[(data["Residue"] >= start) & (data["Residue"] <= end)]
        
        values = seg_data["Average Propensity"].values
        stds   = seg_data["Standard Deviation"].values
        
        N = len(values)
        
        mean_value = np.mean(values)
        
        # Proper error propagation
        sem_value = np.sqrt(np.sum(stds**2)) / N
        
        segment_means[seg_name].append(mean_value)
        segment_sems[seg_name].append(sem_value)

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.tab10(np.linspace(0, 1, len(segments)))

for (seg_name, means), color in zip(segment_means.items(), colors):
    ax.errorbar(
        hbs_values,
        means,
        yerr=segment_sems[seg_name],
        fmt='o-',
        linewidth=2.5,
        markersize=7,
        capsize=4,
        label=seg_name,
        color=color
    )

# Formatting
ax.set_xlabel("H-bond Strength (%)", fontsize=30, fontweight='bold')
ax.set_ylabel("Helical Propensity", fontsize=30, fontweight='bold')

ax.tick_params(axis='both', labelsize=26)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

ax.set_ylim(0, 1.0)

legend = ax.legend( fontsize=26, title_fontsize=18, frameon=False)
for text in legend.get_texts():
    text.set_fontweight('bold')
legend.get_title().set_fontweight('bold')

# Full box with thicker border
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.rcParams['axes.linewidth'] = 2

plt.tight_layout()

#output_path = os.path.join( "IM30 segment_average_vs_HBS_with_error.png")
#plt.savefig(output_path, dpi=600)
print(f"Plot saved to: {output_path}")

plt.show()
